# Pipeline de Dados

Objetivo: transformar os dados brutos em dados limpos, padronizados e confiáveis.


In [0]:
CATALOG = "workspace"
from pyspark.sql import functions as F, Window

> Transformação e tratamento da tabela title_basics para a camada Silver

Remove duplicidades, converte tipos de dados, transforma o indicador de conteúdo adulto em booleano, estrutura os gêneros em formato de lista, padroniza os tipos de produção e filtra apenas filmes, séries e episódios.

In [0]:
bronze_basics = spark.table(f"{CATALOG}.bronze.title_basics")

silver_basics = (
    bronze_basics
    .dropDuplicates(["tconst"])
    .withColumn("startYear", F.expr("try_cast(startYear as int)"))
    .withColumn("endYear", F.expr("try_cast(endYear as int)"))
    .withColumn("runtimeMinutes", F.expr("try_cast(runtimeMinutes as int)"))
    .withColumn("isAdult", F.col("isAdult") == "1")
    .withColumn(
        "genres_array",
        F.split(F.coalesce(F.col("genres"), F.lit("")), ",")
    )
    .withColumn(
        "title_type_pt",
        F.when(F.col("titleType").isin("movie", "tvMovie"), "Filme")
         .when(F.col("titleType").isin("tvSeries", "tvMiniSeries"), "Série")
         .when(F.col("titleType") == "tvEpisode", "Episódio")
         .otherwise("Outro")
    )
    .filter(
        F.col("titleType").isin(
            "movie",
            "tvMovie",
            "tvSeries",
            "tvMiniSeries",
            "tvEpisode"
        )
    )
    .select(
        "tconst",
        "titleType",
        "title_type_pt",
        "primaryTitle",
        "originalTitle",
        "isAdult",
        "startYear",
        "endYear",
        "runtimeMinutes",
        "genres_array"
    )
)

silver_basics.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{CATALOG}.silver.title_basics")

print(
    f"silver.title_basics: "
    f"{spark.table(f'{CATALOG}.silver.title_basics').count():,} linhas"
)

silver.title_basics: 11,204,178 linhas


> Transformação e tratamento da tabela title_ratings

In [0]:
bronze_ratings = spark.table(f"{CATALOG}.bronze.title_ratings")

silver_ratings = (
    bronze_ratings
    .dropDuplicates(["tconst"])
    .withColumn("averageRating", F.col("averageRating").cast("double"))
    .withColumn("numVotes", F.col("numVotes").cast("int"))
    .filter(F.col("averageRating").between(0, 10))
    .filter(F.col("numVotes") > 0)
    .select("tconst", "averageRating", "numVotes")
)

silver_ratings.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{CATALOG}.silver.title_ratings")

print(
    f"silver.title_ratings: "
    f"{spark.table(f'{CATALOG}.silver.title_ratings').count():,} linhas"
)

silver.title_ratings: 1,714,406 linhas


> Consolidação dos títulos com avaliações

In [0]:
silver_title_rated = (
    silver_basics
    .join(
        silver_ratings,
        on="tconst",
        how="inner"
    )
)

silver_title_rated.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{CATALOG}.silver.title_rated")

print(
    f"silver.title_rated: "
    f"{spark.table(f'{CATALOG}.silver.title_rated').count():,} linhas"
)

silver.title_rated: 1,432,172 linhas


> Tratamento da tabela title_principals e filtragem dos participantes dos títulos analisados

In [0]:
bronze_principals = spark.table(
    f"{CATALOG}.bronze.title_principals"
)

titles_in_scope = silver_title_rated.select("tconst")

silver_principals = (
    bronze_principals
    .withColumn(
        "ordering",
        F.col("ordering").cast("int")
    )
    .join(
        F.broadcast(titles_in_scope),
        on="tconst",
        how="inner"
    )
    .dropDuplicates(
        ["tconst", "nconst", "ordering", "category"]
    )
    .select(
        "tconst",
        "ordering",
        "nconst",
        "category",
        "job",
        "characters"
    )
)

silver_principals.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{CATALOG}.silver.title_principals")

print(
    f"silver.title_principals: "
    f"{spark.table(f'{CATALOG}.silver.title_principals').count():,} linhas"
)

silver.title_principals: 20,745,243 linhas


> Tratamento dos dados do elenco

In [0]:
bronze_names = spark.table(
    f"{CATALOG}.bronze.name_basics"
)

cast_people = (
    silver_principals
    .filter(
        F.col("category").isin(
            "actor",
            "actress"
        )
    )
    .select("nconst")
    .distinct()
)

silver_cast_people = (
    bronze_names
    .dropDuplicates(["nconst"])
    .withColumn(
        "birthYear",
        F.col("birthYear").cast("int")
    )
    .join(
        F.broadcast(cast_people),
        on="nconst",
        how="inner"
    )
    .select(
        "nconst",
        "primaryName",
        "birthYear"
    )
)

silver_cast_people.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{CATALOG}.silver.cast_people")

print(
    f"silver.cast_people: "
    f"{spark.table(f'{CATALOG}.silver.cast_people').count():,} linhas"
)

silver.cast_people: 1,279,523 linhas


> Criação da tabela de relacionamento Título × Elenco

Tabela de relacionamento entre títulos e atores/atrizes, registrando os personagens interpretados e sua ordem nos créditos.

In [0]:
silver_cast = (
    silver_principals
    .filter(
        F.col("category").isin(
            "actor",
            "actress"
        )
    )
    .join(
        silver_cast_people.select(
            "nconst",
            "primaryName"
        ),
        on="nconst",
        how="inner"
    )
    .select(
    "tconst",
    "nconst",
    "characters",
    "ordering"
)
    )
)

silver_cast.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{CATALOG}.silver.title_cast")

print(
    f"silver.title_cast: "
    f"{spark.table(f'{CATALOG}.silver.title_cast').count():,} linhas"
)


silver.title_cast: 10,028,177 linhas


> Tratamento da tabela title_episode

In [0]:
bronze_episode = spark.table(
    f"{CATALOG}.bronze.title_episode"
)

silver_episode = (
    bronze_episode
    .withColumn(
        "seasonNumber",
        F.col("seasonNumber").cast("int")
    )
    .withColumn(
        "episodeNumber",
        F.col("episodeNumber").cast("int")
    )
    .join(
        F.broadcast(
            titles_in_scope.withColumnRenamed(
                "tconst",
                "parentTconst"
            )
        ),
        on="parentTconst",
        how="inner"
    )
    .dropDuplicates(["tconst"])
    .select(
        "tconst",
        "parentTconst",
        "seasonNumber",
        "episodeNumber"
    )
)

silver_episode.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{CATALOG}.silver.title_episode")

print(
    f"silver.title_episode: "
    f"{spark.table(f'{CATALOG}.silver.title_episode').count():,} linhas"
)

silver.title_episode: 7,337,797 linhas


> Validação das tabelas Silver

In [0]:
for t in [
    "title_basics",
    "title_ratings",
    "title_rated",
    "title_principals",
    "cast_people",
    "title_cast",
    "title_episode"
]:
    n = spark.table(
        f"{CATALOG}.silver.{t}"
    ).count()

    print(
        f"{CATALOG}.silver.{t}: {n:,} linhas"
    )

workspace.silver.title_basics: 11,204,178 linhas
workspace.silver.title_ratings: 1,714,406 linhas
workspace.silver.title_rated: 1,432,172 linhas
workspace.silver.title_principals: 20,745,243 linhas
workspace.silver.cast_people: 1,279,523 linhas
workspace.silver.title_cast: 10,028,177 linhas
workspace.silver.title_episode: 7,337,797 linhas
